# LFW 04. Probe search and certification

## 예상 소요 시간

| 실행 모드 | 예상 시간 | 주로 오래 걸리는 구간 |
| --- | ---: | --- |
| `EXECUTE_STAGE=False` | 1초 미만 | run 연결과 경로 설정 확인 |
| `EXECUTE_STAGE=True` | LFW 기준 약 1~10분 | probe×gallery 검색과 인증 feature 계산 |

> 검색은 대략 probe 수 × gallery 수에 비례합니다. 실제 시간은 DB 상태와 검색 방식에 따라 달라집니다. 30초마다 heartbeat를 출력합니다.

목표: registered/known-unknown/unknown-unknown probe를 검색하고, 압축 각도 오차로 accept/reject/defer 결정을 인증합니다. 성공 기준은 candidate scope와 gallery size가 명시되고, exact fallback을 포함한 probe별 결과가 기록되는 것입니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 03까지 완료된 같은 `RUN_DIR`와 고정된 probes/templates 파일을 사용합니다. 중단되면 04 전체를 다시 실행해 새 attempt artifact를 만듭니다. candidate set, gallery size, threshold, probe/template 파일이 바뀌면 04를 새 attempt로 실행하고, upstream vector가 바뀌면 해당 노트북부터 다시 시작합니다.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = False
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
LEGACY_RUN_ROOT = PROJECT_ROOT / 'runs'
try:
    RUN_DIR = resolve_active_run(RUN_ROOT)
except FileNotFoundError:
    RUN_ROOT = LEGACY_RUN_ROOT
    RUN_DIR = resolve_active_run(RUN_ROOT)
PROGRESS = ProgressReporter('04 probe search/certification', heartbeat_seconds=30)
PROBES_CSV_VALUE = os.environ.get('RONBUN_PROBES_CSV', '').strip()
TEMPLATES_CSV_VALUE = os.environ.get('RONBUN_TEMPLATES_CSV', '').strip()


## Plan

- Require explicit probe/template files with JSON-array embedding columns.
- Keep the three open-set probe types separate and validate exhaustive versus candidate-set scope.
- Save certified features and concise coverage/defer/fallback summaries per attempt.


In [ ]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable.')
    return run, manifest

def resolve_input(value: str, configured: str | None) -> Path | None:
    selected = value or (configured or '')
    if not selected:
        return None
    path = Path(selected)
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()

preflight = {'execute_stage': EXECUTE_STAGE, 'run_dir_resolved': str(RUN_DIR),
             'probes_override_supplied': bool(PROBES_CSV_VALUE), 'templates_override_supplied': bool(TEMPLATES_CSV_VALUE)}
preflight


## Execute and record

`candidate_scope=candidate_set` 결과는 전체 gallery에 대한 global certification으로 주장할 수 없습니다. 전역 인증은 exhaustive 후보와 정확한 gallery size가 일치할 때만 기록합니다.


In [ ]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    PROGRESS.emit('실행 시작', expected='LFW 기준 약 1~10분; 입력 크기에 비례')
    import pandas as pd
    from research.search.open_set import build_certified_search_features, summarize_certified_search_features

    with PROGRESS.step('run/input 및 03 artifact 검증', expected='10초 미만'):
        run, run_manifest = attach_run(RUN_DIR)
        run.verify_inputs()
        run.verify_phase_artifacts('03_compressed_materialization_and_index')
    config = run_manifest['config']
    search_cfg = config.get('search', {})
    probes_path = resolve_input(PROBES_CSV_VALUE, search_cfg.get('probes_path'))
    templates_path = resolve_input(TEMPLATES_CSV_VALUE, search_cfg.get('templates_path'))
    if probes_path is None or templates_path is None:
        raise ValueError('Set RONBUN_PROBES_CSV and RONBUN_TEMPLATES_CSV or configure search paths.')
    if not probes_path.is_file() or not templates_path.is_file():
        raise FileNotFoundError({'probes': str(probes_path), 'templates': str(templates_path)})
    registered_inputs = json.loads(run.manifest_path.read_text(encoding='utf-8')).get('inputs', [])
    for role, path in (('probe_vectors', probes_path), ('template_vectors', templates_path)):
        same_role = [entry for entry in registered_inputs if entry.get('role') == role]
        if same_role and all(Path(entry['path']).resolve() != path.resolve() for entry in same_role):
            raise ValueError(f'{role} already points to another file; start a new run from notebook 00.')
        run.record_input(path, role=role)
    run.verify_inputs(roles={'probe_vectors', 'template_vectors'})

    def read_vectors(path: Path) -> pd.DataFrame:
        frame = pd.read_csv(path)
        for column in ('embedding', 'fallback_embedding'):
            if column in frame.columns:
                frame[column] = frame[column].map(lambda value: json.loads(value) if isinstance(value, str) else value)
        return frame

    with run.phase('04_probe_search_and_certification') as phase:
        with PROGRESS.step('probe/template CSV 로딩', expected='10초~1분'):
            probes = read_vectors(probes_path)
            templates = read_vectors(templates_path)
        PROGRESS.emit('검색 입력 준비 완료', probes=len(probes), templates=len(templates))
        required_probe_types = {'registered', 'known_unknown', 'unknown_unknown'}
        missing_types = sorted(required_probe_types.difference(set(probes['probe_type'].astype(str))))
        if missing_types:
            raise ValueError(f'Missing probe types: {missing_types}')
        scope = str(search_cfg.get('candidate_scope', 'exhaustive'))
        configured_gallery_size = search_cfg.get('gallery_size')
        gallery_size = int(configured_gallery_size) if configured_gallery_size is not None else len(templates)
        with PROGRESS.step('probe×gallery 검색 및 certification 계산', expected='1분 이상; O(probes×gallery)'):
            features = build_certified_search_features(
                probes, templates,
                compression_profile=str(search_cfg.get('compression_profile', 'pca_256')),
                threshold=float(config['certification']['threshold']),
                top_k=int(search_cfg.get('top_k', 2)), candidate_scope=scope, gallery_size=gallery_size,
            )
        summary = summarize_certified_search_features(features)
        PROGRESS.emit('검색 완료, artifact 저장 시작', certified_rows=len(features))
        suffix = f'A{phase.attempt:03d}'
        features_source = phase.attempt_dir / f'certified_features_{suffix}.csv'
        summary_source = phase.attempt_dir / f'certification_summary_{suffix}.json'
        features.to_csv(features_source, index=False)
        summary_source.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
        phase.publish_artifact(features_source)
        phase.publish_artifact(summary_source)
        phase.record_counts(probes=len(probes), templates=len(templates), certified_rows=len(features))
        phase.record('certification_scope', candidate_scope=scope, gallery_size=gallery_size, global_claim=bool(features['certification_global_claim'].all()))
    PROGRESS.emit('04 완료', probes=len(probes), templates=len(templates), certified_rows=len(features))
    result = {'status': 'completed', 'run_id': run.run_id, 'candidate_scope': scope, 'gallery_size': gallery_size, **summary}
else:
    PROGRESS.emit('검토 모드 완료: 검색과 certification을 실행하지 않음', expected='1초 미만')
result


## Next step

probe type별 certification coverage, defer rate, exact fallback rate를 확인한 뒤 05로 이동합니다. candidate scope를 바꾼 결과는 같은 표에서 전역 인증 결과처럼 섞지 않습니다.
